In [ ]:
# max001

In [ ]:
# conda env: scib_no_defaults

In [ ]:
# we use asw_label to compute "original" silhouette for the original scenarios, reversing the transformation to range between 0-1 back to the original -1 to 1 range.

In [1]:
import scanpy as sc
import anndata as ad
import scib
import numpy as np
import pandas as pd

In [2]:
%run ./../custom_silhouette_functions.ipynb

Signature:
silhouette_samples_custom(
    X,
    labels,
    metric='euclidean',
    between_cluster_distances='nearest',
)
Docstring:
Compute the average silhouette score for the dataset X with the given labels.

Parameters:
X : array-like, shape (n_samples, n_features)
    Feature array.
labels : array-like, shape (n_samples,)
    Labels of each point.
    
metric : metric for distance calculation, default:"euclidean", alternatives, e.g., "cosine"

between_cluster_distances: one out of "mean_other", "furthest", "nearest"


Returns:
score : float
    The average silhouette score.
File:      /tmp/7502983.1.gpu.q/ipykernel_2876154/4094074416.py
Type:      function

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [4]:
scenarios = ['original_k2', 'original_k3', 'original_k4',
             'bio_cons_distance_good', 'bio_cons_distance_worse', 'bio_cons_distance_best',
            'bio_cons_shape_overlap', 'bio_cons_shape_batch_effects', 'bio_cons_shape_odd', 
            'batch_removal_nested_Strong', 'batch_removal_nested_Mild', 'batch_removal_nested_None',
            'batch_removal_global_Strong', 'batch_removal_global_None']

In [18]:
np.random.seed(61)

# Collect computed scores, nested dict is simple to convert to pd.DataFrame
score_dict = {}
for scenario in scenarios:
    # Initialize nested dict
    score_dict[scenario] = {}
    
    adata = ad.AnnData(X=pd.read_csv('./../../data/simulated_2d/{}.csv'.format(scenario))[['X', 'Y']].values, obs=pd.read_csv('./../../data/simulated_2d/{}.csv'.format(scenario))[['sample', 'cell_type']])
    adata.obs['Batch'] = adata.obs['sample'].astype('category')

    if 'global' in scenario:
        adata.obs['cell_type'] = "dummy"

    adata.obs['Cell_type'] = adata.obs['cell_type'].astype('category')

    adata.obsm['XY'] = adata.X
    sc.pp.neighbors(adata, use_rep='XY')

    # Compute scores
    ## Level of evaluation: batch/sample
    ### asw_batch
    score = scib.me.silhouette_batch(
        adata,
        batch_key='Batch',
        group_key='Cell_type',
        embed='XY',
        verbose=False
    )
    score_dict[scenario]['asw_batch'] = score
    
    score = scib.me.silhouette_batch(
        adata,
        batch_key='Batch',
        group_key='Cell_type',
        embed='XY',
        metric='cosine',
        verbose=False
    )
    score_dict[scenario]['asw_batch_cosine'] = score
    
    
    ### asw_batch_mean_other
    score = silhouette_batch_custom(
        adata,
        batch_key='Batch',
        group_key='Cell_type',
        embed='XY',
        between_cluster_distances='mean_other',
        verbose=False
    )
    score_dict[scenario]['asw_batch_mean_other'] = score
    
    score = silhouette_batch_custom(
        adata,
        batch_key='Batch',
        group_key='Cell_type',
        embed='XY',
        between_cluster_distances='mean_other',
        metric='cosine',
        verbose=False
    )
    score_dict[scenario]['asw_batch_mean_other_cosine'] = score
    
    ### asw_batch_furthest
    score = silhouette_batch_custom(
        adata,
        batch_key='Batch',
        group_key='Cell_type',
        embed='XY',
        between_cluster_distances='furthest',
        verbose=False
    )
    score_dict[scenario]['asw_batch_furthest'] = score
    
    score = silhouette_batch_custom(
        adata,
        batch_key='Batch',
        group_key='Cell_type',
        embed='XY',
        between_cluster_distances='furthest',
        metric='cosine',
        verbose=False
    )
    score_dict[scenario]['asw_batch_furthest_cosine'] = score
    
    ### graph iLISI and cLISI on variable batch
    score_dict[scenario]['iLISI_batch'], score_dict[scenario]['cLISI_full'] =  scib.me.lisi.lisi_graph(adata, batch_key='Batch', label_key='Cell_type', type_='knn')
        
    means = []
    total = 0
    for cell_type in adata.obs['Cell_type'].unique():
        tmp_adata = adata[adata.obs['Cell_type']==cell_type]
        cell_type_iLISI = scib.metrics.ilisi_graph(tmp_adata, batch_key='Batch', type_='knn')
        means += [cell_type_iLISI * tmp_adata.shape[0]]
        total += tmp_adata.shape[0]
        print(cell_type, cell_type_iLISI)
    print(means)
    print(np.nansum(means)/total)
    score_dict[scenario]['CiLISI_batch'] = np.nansum(means)/total
    
    ### asw_label
    try:
        score = scib.me.silhouette(
            adata,
            group_key='Cell_type',
            embed='XY',
        )
        score_dict[scenario]['asw_label'] = score
    
    
        score = scib.me.silhouette(
            adata,
            group_key='Cell_type',
            embed='XY',
            metric='cosine'
        )
        score_dict[scenario]['asw_label_cosine'] = score

        ### nmi    
        scib.metrics.cluster_optimal_resolution(
            adata,
            resolutions=np.arange(0.01, 0.21, 0.01).tolist(),
            label_key='Cell_type',
            cluster_key='cluster',
            metric=scib.me.nmi
        )
        
        score = scib.me.nmi(
            adata,
            group1='cluster',
            group2='Cell_type'
        )
        
        score_dict[scenario]['nmi'] = score
        
        ### ari
        scib.metrics.cluster_optimal_resolution(
            adata,
            resolutions=np.arange(0.01, 0.21, 0.01).tolist(),
            label_key='Cell_type',
            cluster_key='cluster',
            metric=scib.me.ari
        )
        
        score = scib.me.ari(adata, cluster_key="cluster", label_key="Cell_type")
        score_dict[scenario]['ari'] = score

    except:
        score_dict[scenario]['asw_label'] = np.nan
        score_dict[scenario]['asw_label_cosine'] = np.nan
        score_dict[scenario]['nmi'] = np.nan
        score_dict[scenario]['ari'] = np.nan

    

/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/scib/metrics/lisi.py:115: RuntimeWarning: invalid value encountered in scalar divide
  ilisi = (ilisi - 1) / (nbatches - 1)
/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/scib/metrics/lisi.py:115: RuntimeWarning: invalid value encountered in scalar divide
  ilisi = (ilisi - 1) / (nbatches - 1)


Cluster 2 nan


/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/scib/metrics/lisi.py:115: RuntimeWarning: invalid value encountered in scalar divide
  ilisi = (ilisi - 1) / (nbatches - 1)


Cluster 1 nan
[nan, nan]
0.0
resolution: 0.01, nmi: 0.73376472327949
resolution: 0.02, nmi: 0.73376472327949
resolution: 0.03, nmi: 0.73376472327949
resolution: 0.04, nmi: 0.73376472327949
resolution: 0.05, nmi: 0.73376472327949
resolution: 0.060000000000000005, nmi: 0.73376472327949
resolution: 0.06999999999999999, nmi: 0.73376472327949
resolution: 0.08, nmi: 0.6476252623024114
resolution: 0.09, nmi: 0.5492526302912645
resolution: 0.09999999999999999, nmi: 0.5246628078681567
resolution: 0.11, nmi: 0.4992839856326342
resolution: 0.12, nmi: 0.4744375688550883
resolution: 0.13, nmi: 0.4531509636726128
resolution: 0.14, nmi: 0.4721291928650944
resolution: 0.15000000000000002, nmi: 0.45153689574225186
resolution: 0.16, nmi: 0.43504289398228624
resolution: 0.17, nmi: 0.43691745677194366
resolution: 0.18000000000000002, nmi: 0.43691745677194366
resolution: 0.19, nmi: 0.43691472607638604
resolution: 0.2, nmi: 0.4364731122857724
optimised clustering against Cell_type
optimal cluster resolution

/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/scib/metrics/lisi.py:115: RuntimeWarning: invalid value encountered in scalar divide
  ilisi = (ilisi - 1) / (nbatches - 1)
/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/scib/metrics/lisi.py:115: RuntimeWarning: invalid value encountered in scalar divide
  ilisi = (ilisi - 1) / (nbatches - 1)


...

Cell type 1 0.8127402081136723
[3250.9608324546894]
0.8127402081136723


/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/scib/metrics/lisi.py:197: RuntimeWarning: invalid value encountered in scalar divide
  clisi = (nlabs - clisi) / (nlabs - 1)


dummy 0.0
[0.0]
0.0


/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/fast/AG_Ohler/prauten/conda_envs/scib_no_defaults/lib/python3.10/site-packages/scib/metrics/lisi.py:197: RuntimeWarning: invalid value encountered in scalar divide
  clisi = (nlabs - clisi) / (nlabs - 1)


dummy 0.6235892242890833
[2494.356897156333]
0.6235892242890833


In [10]:
pd.options.display.float_format = "{:,.2f}".format

In [20]:
scores = pd.DataFrame(score_dict)
scores

,original_k2,original_k3,original_k4,bio_cons_distance_good,bio_cons_distance_worse,bio_cons_distance_best,bio_cons_shape_overlap,bio_cons_shape_batch_effects,bio_cons_shape_odd,batch_removal_nested_Strong,batch_removal_nested_Mild,batch_removal_nested_None,batch_removal_global_Strong,batch_removal_global_None
asw_batch,NaN,NaN,NaN,NaN,NaN,NaN,0.98,0.23,0.99,0.98,0.96,0.98,0.93,0.96
asw_batch_cosine,NaN,NaN,NaN,NaN,NaN,NaN,0.96,0.09,0.98,0.92,0.89,0.97,0.99,0.95
asw_batch_mean_other,NaN,NaN,NaN,NaN,NaN,NaN,0.98,0.19,0.99,0.20,0.68,0.98,0.93,0.96
asw_batch_mean_other_cosine,NaN,NaN,NaN,NaN,NaN,NaN,0.97,0.11,0.98,0.02,0.43,0.98,0.99,0.95
asw_batch_furthest,NaN,NaN,NaN,NaN,NaN,NaN,0.98,0.17,0.99,0.14,0.61,0.98,0.93,0.96
asw_batch_furthest_cosine,NaN,NaN,NaN,NaN,NaN,NaN,0.96,0.11,0.98,0.01,0.39,0.97,0.99,0.95
iLISI_batch,NaN,NaN,NaN,NaN,NaN,NaN,0.86,0.00,0.87,0.31,0.34,0.81,0.00,0.62
cLISI_full,1.00,1.00,1.00,1.00,1.00,1.00,0.86,1.00,1.00,NaN,NaN,NaN,NaN,NaN
CiLISI_batch,0.00,0.00,0.00,0.00,0.00,0.00,0.85,0.00,0.87,0.31,0.34,0.81,0.00,0.62
asw_label,0.86,0.89,0.81,0.86,0.77,0.93,0.65,0.65,0.65,NaN,NaN,NaN,NaN,NaN


In [ ]:
# transform bio-cons asw_label scores back to "original" silhouette 

In [1]:
#k2
0.86*2-1

0.72

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
#k3
0.89*2-1

0.78

In [3]:
#k4
0.81*2-1

0.6200000000000001